# Introduction

Pytorch is an open source machine learning package for Python. It is built around tensors and easily supports gpu and multi-gpu coding. It supports both NVIDIA and AMD gpus.

It can be installed from https://pytorch.org/get-started/locally/. If you have an NVIDIA gpu, select CUDA as your compute platform. For AMD select ROCm.

In [2]:
import torch
import numpy as np

# Tensors

Pytorch tensors are very similar to numpy array but they can run on gpus. 

In [4]:
data = [[1, 2], [3, 4]]
tensor_native = torch.tensor(data)
array_numpy = np.array(data)
#You can even convert an np array into a tensor!
tensor_from_numpy = torch.from_numpy(array_numpy)

print(tensor_native)
print(array_numpy)
print(tensor_from_numpy)

print(type(tensor_native))

tensor([[1, 2],
        [3, 4]])
[[1 2]
 [3 4]]
tensor([[1, 2],
        [3, 4]])
<class 'torch.Tensor'>


An important attribute of tensors is `.device`. This says where the tensor is being stored

In [5]:
print(tensor_native.device)

cpu


We can move tensors (and most things in Pytorch) to gpus and other accelerators using the `.to()` method. To move to an NVIDIA device, use `.to("cuda")`. Note that it takes time to move data between the cpu and gpu. If the data is small, this can actually take more time than is saved by using a gpu!

In [7]:
#This lists the accelerators that are available
torch.accelerator.current_accelerator()

device(type='mps')

Tensors have equivalents of most numpy functions and can be sliced exactly like numpy arrays!

# Dataloaders

A Dataloader is an iterable provided by Pytorch that supports batching, multiprocessing speedup, and memory pinning. These are used when training Neural Networks. In this example we will use the MNist Fashion data. Each element is a matrix-label pair.

In [8]:
from torch.utils.data import Dataset
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

#These are special pytorch functions for downloading this data because it's so common
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])
)

100.0%
100.0%
100.0%
100.0%


In [ ]:
#Note that we have two elements; a matrix and a label (9)
print(training_data[0])

(Image([[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000],
         [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
          0.0000, 0.0000, 0.0000, 0.0039, 0.0000, 0.0000, 0.0510, 0.2863, 0.0000,
          0.0000, 0.0039, 0.0157, 0.0000,

We now create a dataloader from these datasets for use in the next step

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(dataset=training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=True)


# Neural Networks

NN are implemented using Python Classes in Pytorch.

In [12]:
from torch import nn
class NeuralNetwork(nn.Module):
    #Set out the architecture. Note that you could have multiple architectures in the class
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    #All NN must have a forward method. We don't ever call this ourselves!!!
    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [13]:
#Hand piece of code for using gpu
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using mps device


In [18]:
#Create an instance of the model and then move it to the gpu
model = NeuralNetwork()

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    # Set the model to training mode - important for batch normalization and dropout layers
    model.train()
    # Note that we loop through the dataloader
    for batch, (X, y) in enumerate(dataloader):
        # Call the model on the input X
        pred = model(X)
        # Calculate the loss
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        # Update the params in the model
        optimizer.step()
        # Set the gradient to 0 after the pass
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * batch_size + len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")


def test_loop(dataloader, model, loss_fn):
    # Set the model to evaluation mode - important for batch normalization and dropout layers
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    # Evaluating the model with torch.no_grad() ensures that no gradients are computed during test mode
    # also serves to reduce unnecessary gradient computations and memory usage for tensors with requires_grad=True
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")


In [20]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [21]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.314357  [   64/60000]
loss: 2.289135  [ 6464/60000]
loss: 2.274464  [12864/60000]
loss: 2.260976  [19264/60000]
loss: 2.247103  [25664/60000]
loss: 2.225813  [32064/60000]
loss: 2.203071  [38464/60000]
loss: 2.210320  [44864/60000]
loss: 2.172405  [51264/60000]
loss: 2.140692  [57664/60000]
Test Error: 
 Accuracy: 43.1%, Avg loss: 2.144631 

Epoch 2
-------------------------------
loss: 2.128782  [   64/60000]
loss: 2.134330  [ 6464/60000]
loss: 2.113583  [12864/60000]
loss: 2.043479  [19264/60000]
loss: 2.067693  [25664/60000]
loss: 2.033617  [32064/60000]
loss: 1.988150  [38464/60000]
loss: 1.939608  [44864/60000]
loss: 1.920311  [51264/60000]
loss: 1.870965  [57664/60000]
Test Error: 
 Accuracy: 59.4%, Avg loss: 1.862720 

Epoch 3
-------------------------------
loss: 1.837427  [   64/60000]
loss: 1.790825  [ 6464/60000]
loss: 1.762692  [12864/60000]
loss: 1.678095  [19264/60000]
loss: 1.659443  [25664/60000]
loss: 1.615021  [32064/600